# Redes Convolucionales

## ¿Por qué CNNs para imágenes?

**Problema con las capas totalmente conectadas para imágenes:**
- Imagen 224×224×3 = 150,528 neuronas de entrada
- Si la primera capa oculta tiene 1000 neuronas → ¡150M parámetros!
- Sin conciencia de la estructura espacial

**Las CNNs resuelven esto con:**
1. **Conectividad local**: Cada neurona se conecta a una pequeña región
2. **Compartición de pesos**: El mismo Filter se aplica en todas partes
3. **Invarianza a la traslación**: Detecta características sin importar la posición

## Capas Convolucionales

Al clasificar imágenes, la forma más sencilla de abordarlas es entender cada píxel como una *característica* de entrada, siendo la salida de nuestro modelo la clasificación de la imagen basada en el conjunto de píxeles. Esto se vuelve muy ineficiente cuando la imagen es muy grande, ya que el número de parámetros que deben aprenderse es muy grande. Además, la información relevante en una imagen no es solo el valor de cada píxel, sino también las relaciones entre ellos; los píxeles cercanos suelen formar características más complejas que los píxeles distantes.

Las redes convolucionales son una forma de abordar este problema, ya que permiten reducir el número de parámetros a aprender, manteniendo al mismo tiempo la información relevante de la imagen. Esto se logra mediante el uso de Filters que se aplican a la imagen, de modo que se extraen de ella características relevantes.

Una capa convolucional en una red neuronal actúa como una ventana (o Filter) que se desplaza por la imagen escaneando conjuntos de píxeles cercanos y buscando un patrón específico en ellos. De esta manera, se da un primer paso para buscar *características* de mayor abstracción, que luego son utilizadas por las capas posteriores para realizar la clasificación.

Las capas convolucionales están diseñadas para manejar datos con un alto grado de correlación espacial. Son la arquitectura fundamental para las tareas de **visión por computador**. Aunque las CNNs se aplicaron históricamente también al **Procesamiento del Lenguaje Natural (NLP)**, desde la introducción de la arquitectura *Transformer* (2017) han sido casi completamente reemplazadas por modelos basados en atención en ese dominio.


- [Dot CSV - ¡Redes Neuronales Convolucionales! ¿Cómo funcionan?](https://www.youtube.com/watch?v=V8j1oENVz00&ab_channel=DotCSV)
- [Dot CSV - ¡Patrones Extraños dentro de una Red Neuronal!](https://www.youtube.com/watch?v=ysqpl6w6Wzg&ab_channel=DotCSV)
- [Convolution and image filtering](https://programmerclick.com/article/5690865983/)
- [Cezanne Camacho - Convolutional Neural Networks](https://cezannec.github.io/Convolutional_Neural_Networks/)

## Ejemplo de detección de bordes en una imagen

Podemos ver un código sencillo que aplica un Filter de detección de bordes a una imagen. Los bordes detectados son aquellos donde hay un cambio abrupto en la intensidad de los píxeles de la imagen. Es un ejemplo de lo que se puede hacer en las primeras capas de una red convolucional para detectar características de la imagen.

Podemos intuir que los bordes son una característica importante para la clasificación de imágenes.

In [ ]:
from PIL import Image, ImageFilter
from IPython.display import display

image_path = "../img/bridge.png"
image = Image.open(image_path).convert("RGB")
display(image)

filtered_image = image.filter(ImageFilter.Kernel(
    size=(3, 3), # Tamaño del Kernel (la ventana)
    # kernel=[-1, -1, -1, -1, 8, -1, -1, -1, -1], # Valores de la ventana
     kernel=[0, -1, 0, -1, 4, -1, 0, -1, 0], # Otro Filter de bordes
    scale=1 
))
display(filtered_image)

- [How Matrix Filters Work](https://manifold.net/doc/mfd9/how_matrix_filters_work.htm)

En la visión por computador clásica, Filters como este detector de bordes eran **diseñados manualmente** por ingenieros. Sin embargo, el verdadero poder de las Convolutional Neural Networks (CNNs) es que **aprenden estos Filters automáticamente** a través del backpropagation. En lugar de codificar un detector de bordes de forma fija, inicializamos los Filters con valores aleatorios y dejamos que la red encuentre los patrones matemáticamente óptimos para extraer en la tarea dada. ¡Fascinantemente, los Filters en las primeras capas de una CNN entrenada casi siempre evolucionan naturalmente hacia detectores de bordes y colores, porque la red descubre de forma independiente que encontrar bordes es el mejor primer paso para entender una imagen!

## Capas Convolucionales en PyTorch

In [ ]:
# PyTorch Conv2d
conv = nn.Conv2d(
    in_channels=1,    # Canales de entrada (1 para escala de grises, 3 para RGB)
    out_channels=16,  # Número de Filters
    kernel_size=3,    # Tamaño del Filter
    stride=1,         # Tamaño del paso (Stride)
    padding=1         # Mantener el mismo tamaño espacial (Padding)
)

x = torch.randn(1, 1, 28, 28)  # Batch, Canales, Alto, Ancho
output = conv(x)
print(f"Entrada: {x.shape} → Salida: {output.shape}")
print(f"Parámetros: {conv.weight.shape} ({conv.weight.numel()} pesos + {conv.bias.numel()} sesgos)")

Podemos ver cómo se define una capa convolucional 2D, con 16 **Filters**, cada uno usando ventanas/**Kernels** de 3x3, que se desplazarán 1 píxel por paso (el **Stride**) y utiliza un **Padding** de 1 píxel para mantener la imagen de salida del mismo tamaño (ya que el Kernel es 3x3, sin Padding perderíamos una fila o columna en cada lado).

En este gif puedes ver cómo un Kernel 3x3 sin Padding hace que la imagen de salida pierda una línea por cada borde, reduciendo la imagen.

![](../img/tds_2dconv.gif)

Añadir un Padding de 1 podría solucionar eso:

[![](../img/padding.gif)](https://medium.com/@draj0718/zero-padding-in-convolutional-neural-networks-bf1410438e99)

[![](../img/edge_handling.gif)](https://cezannec.github.io/Convolutional_Neural_Networks/)

### Stride, Padding y tamaño de salida

Antes de implementar convoluciones en código, es útil predecir las dimensiones de salida.

Para una dimensión espacial (alto o ancho):

$$
\text{salida} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1
$$

donde:
- \(W\): tamaño de entrada
- \(K\): tamaño del Kernel
- \(P\): Padding
- \(S\): Stride

Intuición:
- Un Kernel \(K\) más grande generalmente disminuye el tamaño de salida.
- Un Padding \(P\) más grande aumenta el tamaño de salida.
- Un Stride \(S\) más grande disminuye el tamaño de salida (submuestreo más agresivo).

En convoluciones 2D, esta fórmula se aplica de forma independiente al alto y al ancho.

En los frameworks de Deep Learning, esta operación es matemáticamente una correlación cruzada, aunque comúnmente se llama Convolution.

In [ ]:
# Fórmula del tamaño de salida: (W - K + 2P) / S + 1
def conv_output_size(input_size, kernel_size, stride=1, padding=0):
    return (input_size - kernel_size + 2 * padding) // stride + 1
examples = [
    {"kernel": 3, "stride": 1, "padding": 0},
    {"kernel": 3, "stride": 1, "padding": 1},
    {"kernel": 3, "stride": 2, "padding": 0},
    {"kernel": 5, "stride": 1, "padding": 0},
]

print("Tabla de comparación para una entrada de 28×28:")
print("kernel  stride  padding  salida")
for case in examples:
    output_size = conv_output_size(28, case["kernel"], case["stride"], case["padding"])
    print(f"{case['kernel']:>6}  {case['stride']:>6}  {case['padding']:>7}  {output_size:>6}")

## [Arquitectura CNN en PyTorch](https://pytorch.org/docs/stable/nn.html#convolution-layers)

El siguiente ejemplo muestra una arquitectura CNN clásica (LeNet) implementada con el módulo `torch.nn` de PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LeNet(nn.Module):

    def __init__(self):
        super().__init__()
        # Asume imágenes de entrada de 32x32 en escala de grises (N, 1, 32, 32)
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 3)
        self.fc1 = nn.Linear(16 * 6 * 6, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        size = x.size()[1:]
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

La primera capa (`torch.nn.Conv2d(1, 6, 5)`) realiza una operación similar a una Convolution 2D sobre la imagen de entrada.

Esta capa espera tensores en formato `N × C × H × W`. En este ejemplo, la entrada esperada es `N × 1 × 32 × 32`:
- El primer argumento (`1`) es el **número de canales de entrada**. Aquí es 1 para imágenes en escala de grises. Para imágenes RGB sería 3.
- El segundo argumento (`6`) es el **número de canales de salida** (Filters). La capa aprende 6 Filters, por lo que produce 6 Feature Maps.
- El tercer argumento (`5`) es el **tamaño del Kernel**, así que cada Filter usa una ventana de 5×5.

La salida de la primera Convolution tiene forma `N × 6 × 28 × 28`.
Luego ReLU y Max Pooling la reducen a `N × 6 × 14 × 14`.

La siguiente capa convolucional, `conv2 = nn.Conv2d(6, 16, 3)`, espera 6 canales de entrada y produce 16 canales de salida:
- `N × 6 × 14 × 14` → `N × 16 × 12 × 12` (después de `conv2`)
- `N × 16 × 12 × 12` → `N × 16 × 6 × 6` (después de Max Pooling)

Antes de pasar esta salida a las capas lineales, se remodela a `N × (16 * 6 * 6) = N × 576`.

### Seguimiento de formas en LeNet (para entrada `N × 1 × 32 × 32`)

| Etapa | Operación | Forma de salida |
|---|---|---|
| Entrada | — | `N × 1 × 32 × 32` |
| `conv1` | `Conv2d(1, 6, 5)` | `N × 6 × 28 × 28` |
| `pool1` | `MaxPool2d(2)` | `N × 6 × 14 × 14` |
| `conv2` | `Conv2d(6, 16, 3)` | `N × 16 × 12 × 12` |
| `pool2` | `MaxPool2d(2)` | `N × 16 × 6 × 6` |
| aplanar | reshape | `N × 576` |
| `fc1` | `Linear(576, 120)` | `N × 120` |
| `fc2` | `Linear(120, 84)` | `N × 84` |
| `fc3` | `Linear(84, 10)` | `N × 10` |

Si usas entradas de `28 × 28`, esta arquitectura necesita adaptación (o un paso de redimensionamiento de la entrada), porque el tamaño aplanado ya no es `16 × 6 × 6`.

## Feature Maps y Canales

Cada capa convolucional produce múltiples **Feature Maps** (canales de salida), cada uno detectando diferentes características.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torchvision import datasets, transforms

transform = transforms.ToTensor()
mnist = datasets.MNIST("./data", train=True, download=True, transform=transform)
sample_image, label = mnist[0]

conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)

with torch.no_grad():
    feature_maps = conv(sample_image.unsqueeze(0))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
all_axes = axes.flat

all_axes[0].imshow(sample_image.squeeze(), cmap="gray")
all_axes[0].set_title(f"Entrada (dígito {label})")
all_axes[0].axis("off")

for filter_index, current_axis in enumerate(all_axes[1:9]):
    current_axis.imshow(feature_maps[0, filter_index].numpy(), cmap="viridis")
    current_axis.set_title(f"Filter {filter_index}")
    current_axis.axis("off")

all_axes[9].axis("off")

plt.suptitle("Feature Maps de la primera capa conv")
plt.tight_layout()
plt.show()

## Capas de Pooling

El **Pooling** submuestrea los Feature Maps, reduciendo el cómputo y proporcionando invarianza a la traslación.

[![](../img/MaxpoolSample2.png)](https://computersciencewiki.org/index.php?title=Max-pooling_/_Pooling)

### ¿Por qué es útil el Max Pooling?

Mientras que las capas convolucionales extraen características, las capas de Max Pooling son esenciales por varias razones:
1. **Reduce el cómputo:** Al submuestrear la dimensión espacial (por ejemplo, reduciendo a la mitad el ancho y el alto), reduce significativamente el número de parámetros y los cómputos posteriores.
2. **Invarianza a la traslación:** Hace que la red sea menos sensible a pequeños desplazamientos o traslaciones en la imagen de entrada. Si una característica se desplaza ligeramente, la operación de Max Pooling probablemente seguirá produciendo el mismo valor activo.
3. **Extrae características importantes:** Al tomar el valor máximo en un parche, selecciona la activación más fuerte, descartando eficazmente los datos de fondo ruidosos o menos relevantes.
4. **Previene el Overfitting:** Reducir la resolución espacial obliga a la red a aprender características más robustas de alto nivel, en lugar de depender excesivamente de las ubicaciones exactas de los píxeles.
5. **Aumenta el campo receptivo:** Como el mapa de salida es más pequeño, las capas convolucionales posteriores efectivamente "ven" un área mucho mayor de la imagen de entrada original.

In [ ]:
# Max Pooling vs Average Pooling
x = torch.tensor([[[[1., 2., 3., 4.],
                    [5., 6., 7., 8.],
                    [9., 10., 11., 12.],
                    [13., 14., 15., 16.]]]])

max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

print(f"Entrada:\n{x.squeeze()}\n")
print(f"Max Pooling (2×2):\n{max_pool(x).squeeze()}\n")
print(f"Avg Pooling (2×2):\n{avg_pool(x).squeeze()}")

## Fuentes

- https://cs50.harvard.edu/ai/2024/weeks/5/
- https://pytorch.org/tutorials/beginner/introyt/modelsyt_tutorial.html
- https://medium.com/thedeephub/convolutional-neural-networks-a-comprehensive-guide-5cc0b5eae175
- https://www.pinecone.io/learn/series/image-search/cnn/
- https://stackoverflow.com/questions/65554032/understanding-convolutional-layers-shapes
- https://dudeperf3ct.github.io/cnn/mnist/2018/10/17/Force-of-Convolutional-Neural-Networks/
- https://poloclub.github.io/cnn-explainer/